In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import os
import shutil
import re
import seaborn as sns

In [61]:
#Define cell parameters
cell_dm_bohr = 19.3217899560
cell_dm_ang = cell_dm_bohr * 0.529177
cell_matrix = np.array([1.00000000000,  0.000000000000,  0.00000000000,
                          -0.500000000000,  0.866025403784,  0.000000000000, 
                          0.000000000000,  0.000000000000,  2.283539330750]).reshape(3,3)
cell_matrix = cell_matrix * cell_dm_ang

#multiplied cell parameters
cell_dm_bohr = 19.3217899560
cell_dm_ang = cell_dm_bohr * 0.529177
cell_matrix_mult = np.array([1.00000000000*2,  0.000000000000,  0.00000000000,
                          -0.500000000000*2,  0.866025403784*2,  0.000000000000, 
                          0.000000000000,  0.000000000000,  2.283539330750]).reshape(3,3)
cell_matrix_mult = cell_matrix_mult * cell_dm_ang

In [62]:
cell_matrix_mult

array([[ 20.44929369,   0.        ,   0.        ],
       [-10.22464684,  17.70960782,   0.        ],
       [  0.        ,   0.        ,  23.34838321]])

In [4]:
def qsline_to_coordinate(coord_line, cell_matrix):
    coord_list = coord_line.split()[1:4]
    atom_type = coord_line.split()[0]
    for n in range(len(coord_list)):
        coord_list[n] = re.sub("d","0",coord_list[n])
        coord_list[n] = float(coord_list[n])
    coord_array = np.array(coord_list).reshape(1,3)
    coord_array_ang = np.dot(coord_array,cell_matrix)
    return (atom_type,coord_array_ang)

In [6]:
main_dir = './'
bader_df = pd.read_csv('./1_Ag_subCu/charge_df.csv')
coordinate_df = open('./1_Ag_subCu/input.in','r')
coord_lines = coordinate_df.readlines()

In [16]:
bader_df

,Unnamed: 0,atom_type,x,y,z,Charge,Min_dist,atomic_volume,surface_atom,bader_charge
0,0,Cu,-0.0072,0.0257,4.1772,19.0306,2.0714,382.5950,0,-0.0306
1,1,Ni,4.7832,0.0495,4.2919,18.0640,2.0447,307.6178,0,-0.0640
2,2,Cu,-0.0051,16.7322,4.1097,19.0190,2.1071,409.7094,0,-0.0190
3,3,Cu,14.4615,0.0492,4.0996,19.0120,2.1051,417.7300,0,-0.0120
4,4,Ni,16.8115,4.3120,4.2676,18.0450,2.0133,345.6806,0,-0.0450
...,...,...,...,...,...,...,...,...,...,...
59,59,Cu,9.6306,8.3510,15.5927,18.9951,2.0929,424.3526,1,0.0049
60,60,Cu,-7.2277,12.5502,15.5512,19.0248,2.0737,389.9148,1,-0.0248
61,61,Ni,-2.2883,12.6139,15.3984,18.0358,1.9919,317.1977,1,-0.0358
62,62,Ni,2.3849,12.3606,15.3958,18.0354,2.0488,282.9653,1,-0.0354


In [42]:
cell_x = cell_matrix[0]
cell_y = cell_matrix[1]
cell_z = cell_matrix[2]
cell_xy = cell_x + cell_y

cellx_bader = bader_df.copy()
cellx_bader['x'] = cellx_bader['x'] + cell_x[0]
cellx_bader

celly_bader = bader_df.copy()
celly_bader['x'] = celly_bader['x'] + cell_y[0]
celly_bader['y'] = celly_bader['y'] + cell_y[1]
celly_bader

cellxy_bader = bader_df.copy()
cellxy_bader['x'] = cellxy_bader['x'] + cell_xy[0]
cellxy_bader['y'] = cellxy_bader['y'] + cell_xy[1]
cellxy_bader

cell_original = bader_df.copy()

repeated_bader_df = pd.concat([cell_original, cellx_bader, celly_bader, cellxy_bader],ignore_index=True)

In [43]:
repeated_bader_df

,Unnamed: 0,atom_type,x,y,z,Charge,Min_dist,atomic_volume,surface_atom,bader_charge
0,0,Cu,-0.007200,0.025700,4.1772,19.0306,2.0714,382.5950,0,-0.0306
1,1,Ni,4.783200,0.049500,4.2919,18.0640,2.0447,307.6178,0,-0.0640
2,2,Cu,-0.005100,16.732200,4.1097,19.0190,2.1071,409.7094,0,-0.0190
3,3,Cu,14.461500,0.049200,4.0996,19.0120,2.1051,417.7300,0,-0.0120
4,4,Ni,16.811500,4.312000,4.2676,18.0450,2.0133,345.6806,0,-0.0450
...,...,...,...,...,...,...,...,...,...,...
251,59,Cu,14.742923,17.205804,15.5927,18.9951,2.0929,424.3526,1,0.0049
252,60,Cu,-2.115377,21.405004,15.5512,19.0248,2.0737,389.9148,1,-0.0248
253,61,Ni,2.824023,21.468704,15.3984,18.0358,1.9919,317.1977,1,-0.0358
254,62,Ni,7.497223,21.215404,15.3958,18.0354,2.0488,282.9653,1,-0.0354


In [78]:
mult_coord_file = open('./1_Ag_subCu/mult_coord.in','w')
for i in range(0,len(repeated_bader_df)):
    atom_type = repeated_bader_df.atom_type[i]
    coord = np.array([repeated_bader_df.x[i], repeated_bader_df.y[i], repeated_bader_df.z[i]])
    fract_pos_atom = np.linalg.lstsq(cell_matrix_mult.T,coord.T,rcond=None)
    fract_pos_atom = fract_pos_atom[0]/2
    line = atom_type + '      ' 
    for i in fract_pos_atom:
        line = line + str(i) + '      '
    mult_coord_file.write(line + '\n')
mult_coord_file.close()
    

In [65]:
pos_atom[0]/2

array([0.00018675, 0.00072559, 0.08945373])

In [51]:
np.matmul([0.0003984294,0.0015372176,0.0946746194],cell_matrix)

array([-0.00378495,  0.01361176,  2.21049929])

In [59]:
print(0.0003984294*2)
print(0.0015372176*2)
print(0.0946746194*2)

0.0007968588
0.0030744352
0.1893492388


In [52]:
coord_lines

['Cu            0.0003984294        0.0015372176        0.0946746194\n',
 'Ni            0.2490339313        0.0029574044        0.0972739505\n',
 'Cu            0.4997073835       -0.0000560392        0.0931433438\n',
 'Cu            0.7499240576        0.0029395687        0.0929141542\n',
 'Ni           -0.0010745390        0.2576915899        0.0967224662\n',
 'Cu            0.2473868979        0.2506653091        0.0948729914\n',
 'Cu            0.5028069548        0.2502669540        0.0937334992\n',
 'Ni            0.7569423728        0.2576424405        0.0972772305\n',
 'Ni           -0.0014652502        0.4963392810        0.0969301587\n',
 'Ni            0.2455246955        0.5024518906        0.0968974449\n',
 'Cu            0.5003371355        0.5018105510        0.0944893670\n',
 'Ni            0.7518635062        0.4963212695        0.0979894200\n',
 'Cu           -0.0006628994        0.7484248852        0.0939367266\n',
 'Cu            0.2481451058        0.7488714384   

In [79]:
cell_dm_bohr = 19.3217899560
cell_dm_ang = cell_dm_bohr * 0.529177
cell_matrix_mult = np.array([1.00000000000*2,  0.000000000000,  0.00000000000,
                          -0.500000000000*2,  0.866025403784*2,  0.000000000000, 
                          0.000000000000,  0.000000000000,  2.283539330750]).reshape(3,3)

In [80]:
cell_matrix_mult

array([[ 2.        ,  0.        ,  0.        ],
       [-1.        ,  1.73205081,  0.        ],
       [ 0.        ,  0.        ,  2.28353933]])